In [1]:
import numpy as np

import torch
from torch import optim

import random
from copy import deepcopy

from RecVAE_utils import get_data, ndcg, recall, implicit_slim
from RecVAE_model import VAE

dataset = 'RecVAE_lastfm_2/'
hidden_dim = 600
latent_dim = 200
batch_size = 500
beta = None
gamma = 0.005
lr = 5e-4
n_epochs =30
n_enc_epochs = 3
n_dec_epochs = 1
not_alternating = False
implicitslim = False
lambd = None
alpha = None
threshold = None
step = None

seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda:0")

data = get_data(dataset)
train_data, valid_in_data, valid_out_data, test_in_data, test_out_data = data


def generate(batch_size, device, data_in, data_out=None, shuffle=False, samples_perc_per_epoch=1):
    assert 0 < samples_perc_per_epoch <= 1
    
    total_samples = data_in.shape[0]
    samples_per_epoch = int(total_samples * samples_perc_per_epoch)
    
    if shuffle:
        idxlist = np.arange(total_samples)
        np.random.shuffle(idxlist)
        idxlist = idxlist[:samples_per_epoch]
    else:
        idxlist = np.arange(samples_per_epoch)
    
    for st_idx in range(0, samples_per_epoch, batch_size):
        end_idx = min(st_idx + batch_size, samples_per_epoch)
        idx = idxlist[st_idx:end_idx]

        yield Batch(device, idx, data_in, data_out)


class Batch:
    def __init__(self, device, idx, data_in, data_out=None):
        self._device = device
        self._idx = idx
        self._data_in = data_in
        self._data_out = data_out
    
    def get_idx(self):
        return self._idx
    
    def get_idx_to_dev(self):
        return torch.LongTensor(self.get_idx()).to(self._device)
        
    def get_ratings(self, is_out=False):
        data = self._data_out if is_out else self._data_in
        return data[self._idx]
    
    def get_ratings_to_dev(self, is_out=False):
        return torch.Tensor(
            self.get_ratings(is_out).toarray()
        ).to(self._device)


def evaluate(model, data_in, data_out, metrics, samples_perc_per_epoch=1, batch_size=500):
    metrics = deepcopy(metrics)
    model.eval()
    
    for m in metrics:
        m['score'] = []
    for batch in generate(batch_size=batch_size,
                          device=device,
                          data_in=data_in,
                          data_out=data_out,
                          samples_perc_per_epoch=samples_perc_per_epoch
                         ):
        ratings_in = batch.get_ratings_to_dev()
        ratings_out = batch.get_ratings(is_out=True)
        ratings_pred = model(ratings_in, calculate_loss=False).cpu().detach().numpy()
        
        if not (data_in is data_out):
            ratings_pred[batch.get_ratings().nonzero()] = -np.inf
            
        for m in metrics:
            m['score'].append(m['metric'](ratings_pred, ratings_out, k=m['k']))
    for m in metrics:
        m['score'] = np.concatenate(m['score']).mean()
        
    return [x['score'] for x in metrics]


def run(model, opts, train_data, batch_size, n_epochs, beta, gamma, dropout_rate):
    model.train()
    for epoch in range(n_epochs):
        for batch in generate(batch_size=batch_size, device=device, data_in=train_data, shuffle=True):
            ratings = batch.get_ratings_to_dev()

            for optimizer in opts:
                optimizer.zero_grad()
                
            _, loss = model(ratings, beta=beta, gamma=gamma, dropout_rate=dropout_rate)
            loss.backward()
            
            for optimizer in opts:
                optimizer.step()


model_kwargs = {
    'hidden_dim': hidden_dim,
    'latent_dim': latent_dim,
    'input_dim': train_data.shape[1]
}
metrics = [{'metric': ndcg, 'k': 100}]

best_ndcg = -np.inf
train_scores, valid_scores = [], []

model = VAE(**model_kwargs).to(device)
model_best = VAE(**model_kwargs).to(device)

learning_kwargs = {
    'model': model,
    'train_data': train_data,
    'batch_size': batch_size,
    'beta': beta,
    'gamma': gamma
}

decoder_params = set(model.decoder.parameters())
encoder_params = set(model.encoder.parameters())

optimizer_encoder = optim.Adam(encoder_params, lr=lr)
optimizer_decoder = optim.Adam(decoder_params, lr=lr)



for epoch in range(n_epochs):

    if implicitslim and epoch % step == step - 1:
        encoder_embs = model.encoder.fc1.weight.data
        decoder_embs = model.decoder.weight.data.T
        for embs in [encoder_embs, decoder_embs]:
            embs[:] = torch.Tensor(
                implicit_slim(embs.detach().cpu().numpy(), train_data, lambd, alpha, threshold)
            ).to(device)
    
    if not_alternating:
        run(opts=[optimizer_encoder, optimizer_decoder], n_epochs=1, dropout_rate=0.5, **learning_kwargs)
    else:
        run(opts=[optimizer_encoder], n_epochs=n_enc_epochs, dropout_rate=0.5, **learning_kwargs)
        model.update_prior()
        run(opts=[optimizer_decoder], n_epochs=n_dec_epochs, dropout_rate=0, **learning_kwargs)

    train_scores.append(
        evaluate(model, train_data, train_data, metrics, 0.01)[0]
    )
    valid_scores.append(
        evaluate(model, valid_in_data, valid_out_data, metrics, 1)[0]
    )
    
    if valid_scores[-1] > best_ndcg:
        best_ndcg = valid_scores[-1]
        model_best.load_state_dict(deepcopy(model.state_dict()))
        

    print(f'epoch {epoch} | valid ndcg@100: {valid_scores[-1]:.4f} | ' +
          f'best valid: {best_ndcg:.4f} | train ndcg@100: {train_scores[-1]:.4f}')


    
test_metrics = [{'metric': ndcg, 'k': 5}, {'metric': recall, 'k': 5}, {'metric': ndcg, 'k': 10}, {'metric': recall, 'k': 10}, {'metric': ndcg, 'k': 20}, {'metric': recall, 'k': 20}]

final_scores = evaluate(model_best, test_in_data, test_out_data, test_metrics)

for metric, score in zip(test_metrics, final_scores):
    print(f"{metric['metric'].__name__}@{metric['k']}:\t{score:.4f}")

epoch 0 | valid ndcg@100: 0.0659 | best valid: 0.0659 | train ndcg@100: 0.2142
epoch 1 | valid ndcg@100: 0.1176 | best valid: 0.1176 | train ndcg@100: 0.3557
epoch 2 | valid ndcg@100: 0.1537 | best valid: 0.1537 | train ndcg@100: 0.4335
epoch 3 | valid ndcg@100: 0.1774 | best valid: 0.1774 | train ndcg@100: 0.4679
epoch 4 | valid ndcg@100: 0.2037 | best valid: 0.2037 | train ndcg@100: 0.4967
epoch 5 | valid ndcg@100: 0.2339 | best valid: 0.2339 | train ndcg@100: 0.5420
epoch 6 | valid ndcg@100: 0.2542 | best valid: 0.2542 | train ndcg@100: 0.5805
epoch 7 | valid ndcg@100: 0.2680 | best valid: 0.2680 | train ndcg@100: 0.6167
epoch 8 | valid ndcg@100: 0.2874 | best valid: 0.2874 | train ndcg@100: 0.6448
epoch 9 | valid ndcg@100: 0.3013 | best valid: 0.3013 | train ndcg@100: 0.6657
epoch 10 | valid ndcg@100: 0.3139 | best valid: 0.3139 | train ndcg@100: 0.6878
epoch 11 | valid ndcg@100: 0.3191 | best valid: 0.3191 | train ndcg@100: 0.7180
epoch 12 | valid ndcg@100: 0.3276 | best valid: 0.

In [2]:
test_metrics = [{'metric': ndcg, 'k': 5}, {'metric': recall, 'k': 5}, {'metric': ndcg, 'k': 10}, {'metric': recall, 'k': 10}, {'metric': ndcg, 'k': 20}, {'metric': recall, 'k': 20}]

final_scores = evaluate(model_best, test_in_data, test_out_data, test_metrics)

for metric, score in zip(test_metrics, final_scores):
    print(f"{metric['metric'].__name__}@{metric['k']}:\t{score:.4f}")

ndcg@5:	0.2785
recall@5:	0.2510
ndcg@10:	0.2459
recall@10:	0.2171
ndcg@20:	0.2932
recall@20:	0.3080
